# Phase 0 — EDA & Recall-Ceiling Probe

Runs the full recall-ceiling probe over the **dev** split (incl. the dense-text channel that needs a GPU encoder), answers the plan §5 EDA questions, and emits the **DESIGN PARAMETERS** block consumed by R7/K1/R1/L1.

Spec: `.claude/documents/features/20_P0_eda_recall_probe.md`. Analysis-only; no training. GPU runtime recommended (dense encode of ~47k docs).

## 1. Setup — clone repo@branch + install

In [ ]:
REPO = 'https://github.com/orrimoch/recsys2026-lora-tutorial.git'
BRANCH = 'fresh-start'
!git clone --branch $BRANCH --depth 1 $REPO recsys2026 2>/dev/null || (cd recsys2026 && git pull)
%cd recsys2026
!pip -q install datasets bm25s scipy sentence-transformers numpy pandas

In [ ]:
import sys; sys.path.insert(0, '.')
# data: download once (or mount Drive). download_data.py pulls the TalkPlayData-Challenge-* dirs into ./data
import os
if not os.path.isdir('data/TalkPlayData-Challenge-Track-Metadata'):
    !python download_data.py

## 2. Config

In [ ]:
DEV_SPLIT = 'test'          # the public dev split (has golds)
DEPTH = 500                 # probe pull depth (>= fusion-K target)
KS = [20, 50, 100, 200, 500]
COLD_THRESHOLD = 1
DENSE_MODEL = 'BAAI/bge-large-en-v1.5'   # R4 encoder (query + catalog docs, same space)
CONTENT_MODALITY = 'metadata-qwen3_embedding_0.6b'  # provided content emb for content-kNN

## 3. Load data (F1)

In [ ]:
from datasets import load_from_disk
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations

cat = Catalog.from_disk('data/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
te_content = TrackEmbeddings.from_disk('data/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks', modalities=[CONTENT_MODALITY])
te_cf = TrackEmbeddings.from_disk('data/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks', modalities=['cf-bpr'])
ue = UserEmbeddings.from_disk('data/TalkPlayData-Challenge-User-Embeddings')
conv = Conversations.from_disk('data/TalkPlayData-Challenge-Dataset', split=DEV_SPLIT, cold_threshold=COLD_THRESHOLD)
print('catalog', len(cat))

## 4. EDA questions (§5.1–5.6, 5.8)
Catalog size/integrity, gold∈history rate, context-length cap, cold/warm split, leakage audit.

In [ ]:
from mcrs.data.ids import canonical_track_id
turns = list(conv.turns())
golds = [conv.gold(t.session_id, t.turn_number) for t in turns]
print('catalog_size =', len(cat), '(47,071 expected)')
print('dev turns =', len(turns))
print('golds_in_catalog =', sum(g in cat for g in golds), '/', len(turns))
# gold-in-history rate -> L1 history-rule default
gih = sum(1 for t,g in zip(turns,golds) if g in set(t.history_tids))
print('gold_in_history_rate =', round(gih/len(turns), 4))
# cold/warm share
segs = [t.segment for t in turns]
print('cold/warm =', segs.count('cold'), '/', segs.count('warm'))
# context length (whitespace tokens of the concatenated causal utterances) -> R1/§8 cap
import numpy as np
ctx_len = np.array([sum(len(u.split()) for u in t.utterances) for t in turns])
print('ctx tokens p50/p90/p95/p99/max =', [int(np.percentile(ctx_len,p)) for p in (50,90,95,99,100)])

In [ ]:
# Leakage audit: Train/Dev session-disjoint
tr = set(load_from_disk('data/TalkPlayData-Challenge-Dataset')['train']['session_id'])
dv = set(t.session_id for t in turns)
print('train∩dev sessions =', len(tr & dv), '(expect 0)')

## 5. Build channels (R1/R3/R4/R5) + run probe (R7 fusion)

In [ ]:
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel

qb = QueryBuilder(context_cap=0)   # set cap from the §4 percentiles once decided
queries = [qb.build(t).text for t in turns]
bc = [{'history_tids': t.history_tids, 'user_id': t.user_id} for t in turns]
uids = [t.user_id for t in turns]

# R4 dense: encode catalog docs + queries with the same encoder (normalized)
model = SentenceTransformer(DENSE_MODEL, device='cuda')
doc_texts = [cat.id_to_metadata(t) for t in cat.index_to_id]
doc_mat = model.encode(doc_texts, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
encode_fn = lambda qs: model.encode(qs, batch_size=256, normalize_embeddings=True)
dense = DenseChannel(cat.index_to_id, doc_mat, encode_fn, normalize=False)  # already normalized

In [ ]:
per = {
  'bm25':        BM25Channel(cat).batch_text_to_item_retrieval(queries, DEPTH),
  'dense':       dense.batch_text_to_item_retrieval(queries, DEPTH),
  'content_knn': ContentKNNChannel(te_content, CONTENT_MODALITY).batch_text_to_item_retrieval(queries, DEPTH, batch_context=bc),
  'cf':          CFChannel(ue, te_cf, 'cf-bpr').batch_text_to_item_retrieval(queries, DEPTH, user_ids=uids),
  'same_artist': SameArtistChannel(cat).batch_text_to_item_retrieval(queries, DEPTH, batch_context=bc),
}

## 6. Recall-ceiling table + saturation + DESIGN PARAMETERS

In [ ]:
from mcrs.eval.probe import recall_ceiling
rep = recall_ceiling(per, golds, ks=KS, segments=segs)
import pandas as pd
rows = []
for lab, e in rep['per_channel'].items():
    rows.append({'channel': lab, **{f'r@{k}': round(e['recall'][k],3) for k in KS}, 'unique': round(e['unique_recall'],3)})
rows.append({'channel': 'FUSED', **{f'r@{k}': round(rep['fused']['recall'][k],3) for k in KS}})
tbl = pd.DataFrame(rows); print(tbl.to_string(index=False))
fused = rep['fused']['recall']
fusion_k = next((k for k in KS if fused[k] >= 0.90), None)
print('\nfused recall@200 =', fused[200], '| recall@500 =', fused[500])
print('smallest K with fused recall>=0.90 (fusion_k):', fusion_k)
print('cold/warm fused r@200:', rep['fused']['by_segment']['cold'][200], rep['fused']['by_segment']['warm'][200])

In [ ]:
# Write reports/eda.md (findings + DESIGN PARAMETERS the downstream modules read)
import os; os.makedirs('reports', exist_ok=True)
with open('reports/eda.md','w') as f:
    f.write('# P0 — EDA & Recall-Ceiling (full)\n\n')
    f.write(tbl.to_markdown(index=False) + '\n\n## DESIGN PARAMETERS\n')
    f.write(f'- fusion_k = {fusion_k} (smallest K with fused recall>=0.90; None => not reached, route to A1/R6)\n')
    f.write(f'- topk_internal = {DEPTH} (>= fusion_k)\n')
    f.write(f'- channel_keep_list = (drop channels with ~0 unique-recall above)\n')
    f.write(f'- cold_threshold = {COLD_THRESHOLD}; cold/warm dev share = {segs.count("cold")}/{segs.count("warm")}\n')
    f.write(f'- gold_in_history_rate = {round(gih/len(turns),4)} -> L1 history-rule default\n')
print('wrote reports/eda.md')

## 7. Gate
If **fused recall@200 ≥ 0.90** (and recall@20 ≥ 0.75): retrieval gate met → proceed to Phase 2 rerank.
If not: the recall wall is the binding constraint → prioritize A1 enrichment / doc2query + R6 extension channels, re-probe.